In [1]:
import numpy as np
import pandas as pd
import os

# 1. 폴더 생성
output_dir = "./sdl"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def generate_raw_items(n_students, n_items, target_mean, target_std, min_val, max_val, is_sum=False):
    """
    논문의 통계치를 바탕으로 학생별 문항 데이터(정수)를 생성하는 함수
    """
    # 1. 학생별 목표 평균 점수 생성 (정규분포 활용)
    scores = np.random.randn(n_students)
    scores = (scores - np.mean(scores)) / np.std(scores) * target_std + target_mean
    
    data = np.zeros((n_students, n_items))
    for i in range(n_students):
        # 2. 목표 합계 계산
        target_sum = scores[i] if is_sum else scores[i] * n_items
            
        # 3. 문항별 무작위 분배 후 반올림 및 범위 제한
        row = np.random.rand(n_items) + 0.5 
        row = (row / np.sum(row)) * target_sum
        row = np.clip(np.round(row), min_val, max_val)
        
        # 4. 오차 조정 (목표 합계에 맞춤)
        diff = int(round(target_sum)) - int(np.sum(row))
        attempts = 0
        while diff != 0 and attempts < 1000:
            idx = np.random.randint(n_items)
            if diff > 0 and row[idx] < max_val:
                row[idx] += 1
                diff -= 1
            elif diff < 0 and row[idx] > min_val:
                row[idx] -= 1
                diff += 1
            attempts += 1
        data[i] = row
    return data.astype(int)

# 설정값: 총 36명 (실험 18, 통제 18), 문항수 15개, 9점 척도 
n_per_group = 18
n_items = 15

# ==========================================
# 2. 자기주도학습(SDL) 데이터 생성 (논문 표 9 통계치 반영) 
# ==========================================
# 실험집단 (Experimental)
sdl_pre_exp = generate_raw_items(n_per_group, n_items, 5.15, 0.81, 1, 9)
sdl_post_exp = generate_raw_items(n_per_group, n_items, 6.34, 0.82, 1, 9)

# 통제집단 (Control)
sdl_pre_ctrl = generate_raw_items(n_per_group, n_items, 5.23, 0.80, 1, 9)
sdl_post_ctrl = generate_raw_items(n_per_group, n_items, 5.41, 0.84, 1, 9)

# ==========================================
# 3. 데이터프레임 구성 및 저장
# ==========================================
def build_sdl_df(group_name, sdl_pre, sdl_post, start_id):
    df = pd.DataFrame()
    df['Student_ID'] = [f'{group_name[0].upper()}{str(i).zfill(2)}' for i in range(start_id, start_id + n_per_group)]
    df['Group'] = group_name
    
    # SDL 사전/사후 문항 (1~15)
    for i in range(n_items):
        df[f'SDL_Pre_Q{i+1}'] = sdl_pre[:, i]
        df[f'SDL_Post_Q{i+1}'] = sdl_post[:, i]
    return df

df_exp = build_sdl_df('Experimental', sdl_pre_exp, sdl_post_exp, 1)
df_ctrl = build_sdl_df('Control', sdl_pre_ctrl, sdl_post_ctrl, 1)

df_final = pd.concat([df_exp, df_ctrl], ignore_index=True)

# CSV 저장
file_path = os.path.join(output_dir, "simulated_sdl_data.csv")
df_final.to_csv(file_path, index=False)

print(f"✅ SDL 데이터 생성 완료! 파일 경로: {file_path}")
print(f"📊 데이터 형태: {df_final.shape} (36명, 32개 컬럼)")

✅ SDL 데이터 생성 완료! 파일 경로: ./sdl/simulated_sdl_data.csv
📊 데이터 형태: (36, 32) (36명, 32개 컬럼)
